## Difference between Log-Loss and MSE

#### **Log-Loss (Cross-Entropy Loss)**
Used for **classification** tasks where the goal is to predict probabilities for discrete classes.

*   **Formula (Binary):** $L(y, \hat{y}) = -(y \log(\hat{y}) + (1 - y) \log(1 - \hat{y}))$
*   **Objective:** Maximizes the likelihood of the correct class. It penalizes confident wrong predictions exponentially.
*   **Gradient ($\frac{\partial L}{\partial \hat{y}}$):**
    *   For a single sample: $\frac{\partial L}{\partial \hat{y}} = \frac{\hat{y} - y}{\hat{y}(1 - \hat{y})}$
    *   When combined with a Sigmoid activation ($\sigma(z)$), the gradient simplifies significantly to: $\frac{\partial L}{\partial z} = \hat{y} - y$. This prevents the "vanishing gradient" problem during backpropagation because the gradient is proportional to the error.
*   **Hessian ($\frac{\partial^2 L}{\partial \hat{y}^2}$):**
    *   $\frac{\partial^2 L}{\partial \hat{y}^2} = \frac{1}{\hat{y}(1 - \hat{y})} - \frac{1}{(1 - \hat{y})^2}$ (simplified contextually).
    *   The Hessian is non-constant and depends on the current prediction, making the loss surface non-quadratic.

#### **MSE (Mean Squared Error)**
Used for **regression** tasks where the goal is to predict continuous numerical values.

*   **Formula:** $L(y, \hat{y}) = (y - \hat{y})^2$
*   **Objective:** Minimizes the squared distance between the predicted value and the actual value.
*   **Gradient ($\frac{\partial L}{\partial \hat{y}}$):**
    *   $\frac{\partial L}{\partial \hat{y}} = -2(y - \hat{y})$
    *   The gradient is linear with respect to the error. If the error is large, the gradient is large; as the error approaches zero, the gradient decreases linearly.
*   **Hessian ($\frac{\partial^2 L}{\partial \hat{y}^2}$):**
    *   $\frac{\partial^2 L}{\partial \hat{y}^2} = 2$
    *   The Hessian is a constant. This indicates that the loss surface is a perfect parabola (quadratic), which is globally convex and has a single, stable minimum.


#### **Summary Comparison**

| Feature | Log-Loss | MSE |
| :--- | :--- | :--- |
| **Task Type** | Classification (Probabilistic) | Regression (Continuous) |
| **Error Penalty** | Exponential for wrong/confident predictions | Quadratic for distance from target |
| **Gradient Behavior** | Linear with error (when paired with Sigmoid/Softmax) | Linear with error |
| **Hessian Behavior** | Variable (Non-constant) | Constant (2) |
| **Loss Surface** | Non-quadratic | Quadratic (Parabolic) |
#
---

### Gradient of Log Loss

For logistic regression with the sigmoid function, the log loss (binary cross-entropy) gradient has a remarkably clean form.

#### Setup

Prediction: $\hat{y} = \sigma(z) = \dfrac{1}{1+e^{-z}}$, where $z = w^Tx + b$

Loss for a single example:
$$L = -\big[y\log(\hat{y}) + (1-y)\log(1-\hat{y})\big]$$

#### **Step 1: Derivative of loss w.r.t. $\hat{y}$**

$$\frac{\partial L}{\partial \hat{y}} = -\frac{y}{\hat{y}} + \frac{1-y}{1-\hat{y}}$$
**Simplified form: (numerator - Gradient; denominator - Hessian**
$$\boxed{\frac{\partial L}{\partial \hat{y}} = \frac{\hat{y} - y}{\hat{y}(1-\hat{y})}}$$

#### **Step 2: Derivative of sigmoid w.r.t. $z$**

The sigmoid has the well-known self-referential derivative:
$$\boxed{\frac{\partial \hat{y}}{\partial z} = \hat{y}(1-\hat{y})}$$

#### **Step 3: Chain rule — this is where it collapses nicely**

$$\frac{\partial L}{\partial z} = \frac{\partial L}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial z} = \left(-\frac{y}{\hat{y}} + \frac{1-y}{1-\hat{y}}\right)\hat{y}(1-\hat{y})$$

Distribute the $\hat{y}(1-\hat{y})$ term:

$$= -y(1-\hat{y}) + (1-y)\hat{y} = -y + y\hat{y} + \hat{y} - y\hat{y} = \hat{y} - y$$

$$\boxed{\frac{\partial L}{\partial z} = \hat{y} - y}$$

#### **Step 4: Gradient w.r.t. weights**

Since $z = w^Tx + b$:

$$\boxed{\frac{\partial L}{\partial w} = (\hat{y} - y)\,x \qquad \frac{\partial L}{\partial b} = \hat{y} - y}$$

For a batch of $n$ examples (matrix form, $X \in \mathbb{R}^{n \times d}$):

$$\nabla_w L = \frac{1}{n}X^T(\hat{y} - y)$$

#### Why this matters (and connects to your XGBoost work)

This "prediction minus label" residual form is exactly the **first-order gradient** term you'd have derived for logistic loss in your XGBoost Taylor-expansion writeup — $\boxed{g_i = \hat{y}_i - y_i}$. The second-order term (Hessian) for log loss is:

$$\boxed{h_i = \hat{y}_i(1-\hat{y}_i)}$$

which is why XGBoost's split-gain formula for classification tasks naturally involves the sigmoid's variance term — same math, different framing (parametric GD vs. gradient-boosted tree splits).

The reason $h_i = \hat{y}_i(1 - \hat{y}_i)$ is used in XGBoost is that the model does not perform updates directly on the **probability** ($\hat{y}$), but rather on the **log-odds** (also called the **logit**), which we will call $z$.

In Gradient Boosting, we model the relationship using the sigmoid function:
$$\hat{y} = \sigma(z) = \frac{1}{1 + e^{-z}}$$

To perform Taylor expansion for the loss function, we must take derivatives with respect to the variable the model actually updates: **$z$**.

#### The Mathematical Derivation

**1. The Loss Function (Log-Loss):**
$$L = -[y \ln(\hat{y}) + (1-y) \ln(1-\hat{y})]$$

**2. The First Derivative (Gradient $g$ with respect to $z$):**
Using the chain rule $\frac{\partial L}{\partial z} = \frac{\partial L}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial z}$:
*   We know $\frac{\partial L}{\partial \hat{y}} = \frac{\hat{y}-y}{\hat{y}(1-\hat{y})}$ (from standard calculus).
*   The derivative of the sigmoid function $\frac{\partial \hat{y}}{\partial z}$ is $\hat{y}(1-\hat{y})$.

Multiplying them together:
$$g = \left( \frac{\hat{y}-y}{\hat{y}(1-\hat{y})} \right) \cdot \hat{y}(1-\hat{y}) = \mathbf{\hat{y} - y}$$
*(This matches your image: $g_i = \hat{y}_i - y_i$)*

**3. The Second Derivative (Hessian $h$ with respect to $z$):**
Now we take the derivative of the gradient $g = \hat{y} - y$ with respect to $z$ using the chain rule again:
$$\frac{\partial^2 L}{\partial z^2} = \frac{\partial}{\partial z}(\hat{y} - y) = \frac{\partial \hat{y}}{\partial z}$$

Since the derivative of the sigmoid function $\sigma(z)$ is $\sigma(z)(1-\sigma(z))$, we get:
$$h = \mathbf{\hat{y}(1-\hat{y})}$$

#### Summary
*   If you take the derivative with respect to **probability ($\hat{y}$)**, the Hessian is $\frac{1}{\hat{y}(1-\hat{y})}$.
*   If you take the derivative with respect to **logit ($z$)**, the Hessian is $\hat{y}(1-\hat{y})$.

**XGBoost uses the logit ($z$) version** because the model's internal "scores" are logits. This makes the math significantly simpler and prevents the "division by zero" issues that would occur if a probability $\hat{y}$ became exactly $0$ or $1$.

#### 3. Summary Comparison

| Feature | Derivative w.r.t. **Probability ($\hat{p}$)** | Derivative w.r.t. **Logit ($z$)** |
| :--- | :--- | :--- |
| **Context** | Standard Calculus / Raw Loss | **XGBoost / Gradient Boosting** |
| **Gradient ($g$)** | $\frac{\hat{p}-y}{\hat{p}(1-\hat{p})}$ | $\hat{p} - y$ |
| **Hessian ($h$)** | $\frac{1}{\hat{p}(1-\hat{p})}$ | $\hat{p}(1-\hat{p})$ |